# Child Mind Sleep States

I'll introduce this notebook as a way to use the KLDivLoss, as well as an architecture that is a mix of GRU cells and the UNET architecture.  It can be improved, and done rather easily.  I will discuss the architecture below.

I have used the Kullback-Leibler divergence loss from torch.  

`L(y_pred, y_true) = y_true * (log y_true - log y_pred)`


My reasoning:

1. It is easily interpretable as a probability distribution
2. Ensembles are also easy to interpret, no matter how differently they are trained.

We are going to predict the the critical points, i.e. where the onset and wakeup events happen.  We will take as the target a gaussian of a tunable width centered around the actual point.

The training code:

https://www.kaggle.com/code/danielphalen/cmss-grunet-train


This, of course, draws heavily from @werus23: 

https://www.kaggle.com/code/werus23/sleep-critical-point-train/notebook

https://www.kaggle.com/code/werus23/sleep-critical-point-infer?scriptVersionId=147143158

Which is based on the wonderful discussion pose of @tolgadincer: 

https://www.kaggle.com/competitions/child-mind-institute-detect-sleep-states/discussion/441470


There could be bugs and other issues in here.  No promises!

In [ ]:
from math import pi, sqrt, exp
import torch
from torch import nn,Tensor
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler

import pandas as pd
import numpy as np
import gc
import math

import datetime
from tqdm.auto import tqdm

import ctypes
import torch
from torch.utils.data import Dataset, DataLoader

import copy

class PATHS:
    MAIN_DIR = "/kaggle/input/child-mind-institute-detect-sleep-states/"
    # CSV FILES : 
    SUBMISSION = MAIN_DIR + "sample_submission.csv"
    TRAIN_EVENTS = MAIN_DIR + "train_events.csv"
    # PARQUET FILES:
    TRAIN_SERIES = MAIN_DIR + "train_series.parquet"
    TEST_SERIES = MAIN_DIR + "test_series.parquet"
    # TEST_SERIES = MAIN_DIR + "train_series.parquet"
    
    WORKING_DIR = '/kaggle/working/'
    
    # Version 3
    MODEL = ['/kaggle/input/zzzs-grunet-train-base-2feats/model_resid_bigru_fold1.pth',
             ]
    
    @staticmethod
    def get_series_filename(series_id):
        f = f'{series_id}_test_series.parquet'
        return PATHS.WORKING_DIR + f
    
class CFG:
    DEMO_MODE = False
    VERBOSE = True
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
def clean_memory():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)
    torch.cuda.empty_cache()
   
    
arch = [(2, 8,  2, 17),
        (8, 32,  2, 11),
        (32, 64, 2, 7)]

in_channels = 2
hidden_size = arch[-1][1]
kernel_size=25
stride = arch[-1][0]
dilation = 1
n_layers = 5
dconv_padding=5
len_mult = 2**len(arch)

## Load data and convert to individual frames for later processing

In [ ]:
t1 = datetime.datetime.now()

In [ ]:
test = pd.read_parquet(PATHS.TEST_SERIES)
series_ids = list(test.series_id.unique())
if len(series_ids) == 3:
    print('Is test script')
    test = pd.read_parquet(PATHS.TRAIN_SERIES)
    series_ids = list(test.series_id.unique())[:10]
    
len(series_ids)

In [ ]:
series_ids

In [ ]:
## Takes about 30 minutes for 277, and 28.2 GB of memory
for series_id in tqdm(series_ids):
    s = test[test.series_id == series_id]
    filename = PATHS.get_series_filename(series_id)
    s.to_parquet(filename)

In [ ]:
del test
clean_memory()

## Functions to produce predictions and datasets

In [ ]:
def get_predictions(res_df, target, SIGMA):
    """Function will take a predicted dataframe, and find local maxima to get event location.  The score is determined by the area under the curve"""
    q = res_df[target].max() * 0.1
    tmp = res_df.loc[res_df[target] > q].copy()
    # print(f'Target max = {q}, len = {len(tmp)}')
    tmp['gap'] = tmp['step'].diff()
    tmp = tmp[tmp['gap'] > 5*5]
    # print(f'Target max = {q}, len = {len(tmp)}')
    res = []
    for i in range(len(tmp) + 1):
        start_i = 0 if i == 0 else tmp['step'].iloc[i-1]
        end_i = tmp['step'].iloc[i] if i < len(tmp) else res_df['step'].max()
        v = res_df.loc[(res_df['step'] > start_i) & (res_df['step'] < end_i)]
        if v[target].max() > q:
            # print('Locate in ', start_i, end_i)
            idx = v.idxmax()[target]
            step = v.loc[idx, 'step']
            span = 3*SIGMA
            score = res_df.loc[(res_df['step'] > step - span) & (res_df['step'] < step + span), target].sum()
            res.append([step, target, score])
            
    return res

In [ ]:
def create_predictions(test_ds, i, net):
    net.eval()
    with torch.no_grad():
        series_id = test_ds.series_ids[i]
        data = test_ds.load_data(series_id)
        X = test_ds[i]
        pred = torch.zeros(X.shape).to(CFG.DEVICE, non_blocking=True)

        h = None

        seq_len = X.shape[0]
        for j in range(0, seq_len, max_chunk_size):
            X_chunk = X[j: j + max_chunk_size].float().to(CFG.DEVICE, non_blocking=True)
            y_pred, h = net(X_chunk, h)
            h = [hi.detach() for hi in h]
            
            # Trim edges where convolution not done well
            v = torch.min(y_pred)
            y_pred[:10, :] = v
            y_pred[-10:, :] = v
            
            pred[j: j+max_chunk_size, :] = y_pred

            del X_chunk, y_pred
        clean_memory()
    res_df = pd.DataFrame(torch.softmax(pred.cpu(), axis=0).numpy(), columns=['wakeup', 'onset'])   
    res_df['step'] = data['step'].values
    return res_df

In [ ]:
class SleepDatasetTrain(Dataset):
    """
    We load the dataset, and select only the longest continuous chunk with events each night
    
    """
    def __init__(
        self,
        series_ids,
        events,
        len_mult,
        continuous = None,
        sigma = None
    ):
        self.series_ids = series_ids
        self.continuous = continuous
        self.len_mult = len_mult
        if events is not None:
            self.events = events
            self.sigma = sigma
        else:
            self.events = None
            self.sigma = None
    
    def load_data(self, series_id):
        filename = PATHS.get_series_filename(series_id)
        data = pd.read_parquet(filename)
        if self.events is not None:
            if self.continuous is not None:
                start, end = self.continuous[series_id]
            else:
                start, end = 0, 1000000
            gap = 6*60*12
            tmp = self.events[(self.events.series_id == series_id) & (self.events.night >= start) & (self.events.night <= end)]
            data = data[(data.step > (tmp.step.min() - gap)) & (data.step < (tmp.step.max() + gap))]
            
            data = data.set_index(['series_id', 'step']).join(tmp.set_index(['series_id', 'step'])[['event', 'night']]).reset_index()
            norm = 1/ np.sqrt(pi / self.sigma)
            for evt in ['wakeup', 'onset']:
                steps = data[data.event == evt]['step'].values
                col = f'{evt}_val'
                data[col] = 0.0
                for i in steps:
                    x = 0.5*((data.step.astype(np.int64) - i)/self.sigma)**2
                    data[col] += np.exp(-x)*norm
                data[col] /= data[col].sum()
                
        n = int((len(data) // len_mult) * len_mult)
        
        return data.iloc[:n]
        
    def __len__(self):
        return len(self.series_ids)

    def __getitem__(self, index):
        series_id = self.series_ids[index]
        data = self.load_data(series_id)
        X = data[['anglez','enmo']].values.astype(np.float32)
        X = torch.from_numpy(X)
        if self.sigma is not None:
            Y = data[['wakeup_val', 'onset_val']].values.astype(np.float32)
            Y = torch.from_numpy(Y)
            return X, Y
        else:
            return X

## Network

I decided that we are detecting edge points, similar to segmentation of images, the a UNET like architecture would be helpful.  Instead of downsampling to get indicators, we let the network learn the convolution.  Since we have wakeup events always after onset events, there is some time component, which is why the GRU cells are in the bottleneck of the UNET.


Possible improvements:

1. The encoder layers and decoder layers could better include more convolutions
2. We can add skip connections.
3. We could just put more indicators in the input.  
4. We can add a time component to the input.
5. We can increase the size of the hidden layers.


The Residual GRU is from 

https://www.kaggle.com/competitions/tlvmc-parkinsons-freezing-gait-prediction/discussion/416410

In [ ]:
# Made using Draw.io
from IPython.display import Image
Image('/kaggle/input/grunet-diagram/grunet.jpg')

In [ ]:
class ResidualBiGRU(nn.Module):
    def __init__(self, hidden_size, n_layers=1, bidir=True):
        super(ResidualBiGRU, self).__init__()

        self.hidden_size = hidden_size
        self.n_layers = n_layers

        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            n_layers,
            batch_first=True,
            bidirectional=bidir,
        )
        dir_factor = 2 if bidir else 1
        self.fc1 = nn.Linear(
            hidden_size * dir_factor, hidden_size * dir_factor * 2
        )
        self.ln1 = nn.LayerNorm(hidden_size * dir_factor * 2)
        self.fc2 = nn.Linear(hidden_size * dir_factor * 2, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)

    def forward(self, x, h=None):
        res, new_h = self.gru(x, h)
        # res.shape = (batch_size, sequence_size, 2*hidden_size)

        res = self.fc1(res)
        res = self.ln1(res)
        res = nn.functional.relu(res)

        res = self.fc2(res)
        res = self.ln2(res)
        res = nn.functional.relu(res)

        # skip connection
        res = res + x

        return res, new_h

class MultiResidualBiGRU(nn.Module):
    def __init__(self, input_size, hidden_size, out_size, n_layers, bidir=True):
        super(MultiResidualBiGRU, self).__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.out_size = out_size
        self.n_layers = n_layers

        self.fc_in = nn.Linear(input_size, hidden_size)
        self.ln = nn.LayerNorm(hidden_size)
        self.res_bigrus = nn.ModuleList(
            [
                ResidualBiGRU(hidden_size, n_layers=1, bidir=bidir)
                for _ in range(n_layers)
            ]
        )
        self.fc_out = nn.Linear(hidden_size, out_size)

    def forward(self, x, h=None):
        # if we are at the beginning of a sequence (no hidden state)
        if h is None:
            # (re)initialize the hidden state
            h = [None for _ in range(self.n_layers)]

        x = self.fc_in(x)
        x = self.ln(x)
        x = nn.functional.relu(x)

        new_h = []
        for i, res_bigru in enumerate(self.res_bigrus):
            x, new_hi = res_bigru(x, h[i])
            new_h.append(new_hi)

        x = self.fc_out(x)
#         x = F.normalize(x,dim=0)
        return x, new_h  # log probabilities + hidden states


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, in_channels, hidden_size, kernel_size, stride, padding, dilation, use_layernorm, print_shape):
        super(EncoderLayer, self).__init__()
        self.conv = nn.Conv1d(in_channels, hidden_size, kernel_size, stride, padding=padding, dilation=1)
        self.ln = nn.LayerNorm(hidden_size) if use_layernorm else None
        self.print_shape = print_shape
        
    def forward(self, x):
        x = self.conv(x.transpose(-1,-2))
        if self.print_shape:
            print('After Conv', x.shape)
        if self.ln is not None:
            x = self.ln(x.transpose(-1, -2))
        else:
            x = x.transpose(-1,-2)
        if self.print_shape:
            print('After Layernorm', x.shape)
        x = nn.functional.relu(x)
        return x

In [ ]:
class GRUNET(nn.Module):
    def __init__(self, arch, out_channels, kernel_size, stride, dconv_padding, hidden_size, n_layers, bidir=True, print_shape=False):
        super(GRUNET, self).__init__()

        self.input_size = in_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.hidden_size = hidden_size
        # self.out_size = out_size
        self.n_layers = n_layers
        self.padding = kernel_size//2
        self.print_shape = print_shape
        self.arch = arch
        self.dilation = 1
        assert arch[-1][1] == hidden_size

        self.conv = nn.Sequential(*[EncoderLayer(in_chan, out_chan, ksize, stride=stride, padding=ksize//2, dilation=self.dilation, use_layernorm=True, print_shape=print_shape) for in_chan, out_chan, stride, ksize in self.arch])
        self.res_bigrus = nn.ModuleList(
            [
                ResidualBiGRU(hidden_size, n_layers=1, bidir=bidir)
                for _ in range(n_layers)
            ]
        )
        self.dconv = nn.Sequential(*sum([[nn.ConvTranspose1d(out_chan, in_chan, ksize, stride=stride, padding=ksize//2, dilation=self.dilation, output_padding=1), 
                                  nn.Conv1d(in_chan, in_chan, ksize, stride=1, padding=ksize//2, dilation=self.dilation), nn.ReLU(), 
                                  nn.Conv1d(in_chan, in_chan, ksize, stride=1, padding=ksize//2, dilation=self.dilation), nn.ReLU()] for in_chan, out_chan, stride, ksize in reversed(arch)], []))
        self.output_layer = nn.Conv1d(2, 2, kernel_size=1, stride=1)
        
    def forward(self, x, h=None):
        # if we are at the beginning of a sequence (no hidden state)
        init_shape = x.shape
        if h is None:
            # (re)initialize the hidden state
            h = [None for _ in range(self.n_layers)]

        if self.print_shape:
            print('In', x.shape)
        x = self.conv(x)
        if self.print_shape:
            print('After EncoderLayer', x.shape)
        new_h = []
        for i, res_bigru in enumerate(self.res_bigrus):
            x, new_hi = res_bigru(x, h[i])
            new_h.append(new_hi)
        if self.print_shape:
            print('After GRU', x.shape)
        x = self.dconv(x.transpose(-1, -2))
        if self.print_shape:
            print('After DConv', x.shape)
            
        x = self.output_layer(x)
        x = x.transpose(-1,-2)
            
        if self.print_shape:
            print('After SmoothConv', x.shape)
        
        return x, new_h  # probabilities + hidden states


In [ ]:
net = GRUNET(arch=arch,out_channels=2, hidden_size=hidden_size, kernel_size=kernel_size, stride=stride, dconv_padding=dconv_padding, n_layers=n_layers, bidir=True, print_shape=True)
X = torch.randn(1040, 2).float()
Z, h = net(X)
assert X.shape == Z.shape

## Inference

In [ ]:
max_chunk_size = 24*60*12  ## Process each day at a time with the network
min_interval = 30  ## minimum sleep interval
SIGMA = 36

In [ ]:
test_ds = SleepDatasetTrain(series_ids, events=None, len_mult=len_mult, sigma=None)

In [ ]:
nets = []
for model in PATHS.MODEL:
    net = GRUNET(arch=arch,out_channels=2, hidden_size=hidden_size, kernel_size=kernel_size, stride=stride, 
                 dconv_padding=dconv_padding, n_layers=n_layers, bidir=True, print_shape=False).to(CFG.DEVICE)
    net.load_state_dict(torch.load(model, map_location=CFG.DEVICE))
    net.eval()
    nets.append(net)

In [ ]:
all_df = []
with torch.no_grad():
    for i in tqdm(range(len(test_ds))):
        series_id = test_ds.series_ids[i]
        # print(series_id)
        data = test_ds.load_data(series_id)
        res_df = None
        for net in nets:
            tmp = create_predictions(test_ds, i, net)
            if res_df is None:
                res_df = tmp
            else:
                res_df['onset'] += tmp['onset']
                res_df['wakeup'] += tmp['wakeup']
                
        res_df['onset'] /= len(nets)
        res_df['wakeup'] /= len(nets)
        onset_pred = get_predictions(res_df, target='onset', SIGMA=SIGMA)
        wakeup_pred = get_predictions(res_df, target='wakeup', SIGMA=SIGMA)
        pred_df = pd.DataFrame(wakeup_pred + onset_pred, columns=['step', 'event', 'score'])
        pred_df['series_id'] = series_id
        pred_df['row_id'] = pred_df.index
        pred_df = pred_df.sort_values(by='step').drop_duplicates(subset='step').reset_index(drop=True)
        min_step, max_step = res_df.step.min(), res_df.step.max()
        pred_df = pred_df[(pred_df.step > min_step + 12*60) & (pred_df.step < max_step - 12*60)]

        all_df.append(pred_df)
        clean_memory()

    pred_df = pd.concat(all_df).reset_index(drop=True)
    pred_df['row_id'] = pred_df.index
    pred_df = pred_df[['row_id', 'series_id', 'step', 'event', 'score']]
    pred_df = pred_df.sort_values(by=['series_id', 'step'])
    

In [ ]:
pred_df[['row_id', 'series_id', 'step', 'event', 'score']].to_csv('submission.csv', index=False)

In [ ]:
pred_df

In [ ]:
!head submission.csv

In [ ]:
t2 = datetime.datetime.now()
print(f'Ran in {t2-t1}')

In [ ]:
from plotly.offline import init_notebook_mode
init_notebook_mode()
from IPython.display import Markdown

import dateutil.relativedelta as rd
from plotly import __version__
from plotly.offline import download_plotlyjs, init_notebook_mode, iplot
from plotly.graph_objs import *
from plotly.tools import FigureFactory as FF

In [ ]:
iplot({'data': [Scatter(x=res_df['step'], y=res_df['wakeup'], name='wakeup'), 
                Scatter(x=res_df['step'], y=res_df['onset'], name='onset')], 
       'layout': Layout(title='Event Probabilities')})